# 04 分红预测误差与基差测量

**研究问题**：用当日可知的预测分红代替事后已实现分红修正基差时，分红预测误差是否会系统性地改变对期货真实升贴水的测量？
这种影响在分红季与非分红季、不同期限的合约之间是否不同？

**为什么能直接研究**：同一天、同一合约下，

```
除预测分红基差 − 除已实现分红基差 = 预测分红 − 已实现分红 = 分红预测误差
```

所以「基差测量误差」就是分红预测误差本身；换成年化口径还要除以指数点位、乘以 365 / 剩余天数，期限越短放大越多。

**样本**：上证50、沪深300、中证500 的四个合约位次。中证1000 没有中证股息点指数，已实现分红无法独立核对，不纳入。
已实现分红要等合约到期才完整，所以到期日晚于样本最后一天的观测剔除。

**读** `basis_daily`（03）、`dividend_point_index`（01）、`forecast_exposure`（02）、`etf_dist_detected`（03）；
**写** `div_error_panel` 与 `research_*` 结果表。不涉及交易策略，也不对套利机会作任何判断。

In [ ]:
import os, sqlite3
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

ROOT = os.getcwd()
DB_PATH = os.path.join(ROOT, "basis.db")
FIG_DIR = os.path.join(ROOT, "results", "figs")
os.makedirs(FIG_DIR, exist_ok=True)

PRODUCTS = {"50": "上证50", "300": "沪深300", "500": "中证500"}
SLOT_LABEL = {"00": "当月", "01": "次月", "02": "季月1", "03": "季月2"}
SEASON_MONTHS = (5, 6, 7, 8)     # 分红季：A 股年报分红集中除息的月份
MIN_DTE_ANN = 10                 # 剩余天数少于 10 天的观测不计年化（×365/DTE 会把小误差放大成噪声）
MATERIAL = 1.0                   # 「实质性误判」门槛：已实现口径年化升贴水绝对值 ≥ 1%

from matplotlib import font_manager
_cjk = [f for f in ["Hiragino Sans GB", "PingFang SC", "Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "Arial Unicode MS"]
        if f in {x.name for x in font_manager.fontManager.ttflist}]
plt.rcParams.update({"font.family": _cjk or ["sans-serif"],
                     "axes.unicode_minus": False, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.color": "#e5e7eb", "grid.linewidth": 0.6})
conn = sqlite3.connect(DB_PATH)

## 一、构建误差面板

每行 = 品种 × 合约位次 × 交易日。`err` 为点数误差（预测 − 已实现），`err_ann` 为年化误差（百分点）。

In [ ]:
b = pd.read_sql("SELECT * FROM basis_daily", conn)
b = b[b["product"].isin(PRODUCTS)].copy()
b["date"] = pd.to_datetime(b["date"])
last_dpi = pd.read_sql("SELECT product, MAX(date) d FROM dividend_point_index GROUP BY product", conn)
last_dpi = dict(zip(last_dpi["product"], pd.to_datetime(last_dpi["d"])))

frames = []
for s in SLOT_LABEL:
    d = pd.DataFrame({"product": b["product"], "date": b["date"], "slot": s, "S": b["spot"], "dte": b[f"{s}_DTE"],
                      "div": b[f"{s}_div"], "div_pred": b[f"{s}_div_pred"],
                      "ann_real": b[f"{s}_ann_exdiv"], "ann_pred": b[f"{s}_ann_exdiv_pred"],
                      "gap": b[f"{s}_basis_exdiv_pred"] - b[f"{s}_basis_exdiv"]})
    frames.append(d)
panel = pd.concat(frames, ignore_index=True).dropna(subset=["S", "dte", "div", "div_pred"])

# 已实现分红要到期才完整：到期日晚于该品种股息点指数最后一天的观测剔除
panel["ltd"] = panel["date"] + pd.to_timedelta(panel["dte"] - 1, unit="D")
panel = panel[panel["ltd"] <= panel["product"].map(last_dpi)].copy()

panel["err"] = panel["div_pred"] - panel["div"]
panel["err_ann"] = panel["err"] / panel["S"] * 365 / panel["dte"] * 100
panel.loc[panel["dte"] < MIN_DTE_ANN, ["err_ann", "ann_real", "ann_pred"]] = np.nan
panel["season"] = panel["date"].dt.month.isin(SEASON_MONTHS)
panel["month"], panel["year"] = panel["date"].dt.month, panel["date"].dt.year

# 恒等式：除预测分红基差 − 除已实现分红基差 = 分红预测误差
assert (panel["gap"] - panel["err"]).abs().max() < 1e-8
ok = panel[["ann_pred", "ann_real", "err_ann"]].notna().all(axis=1)
assert ((panel.loc[ok, "ann_pred"] - panel.loc[ok, "ann_real"]) - panel.loc[ok, "err_ann"]).abs().max() < 1e-8

out = panel.drop(columns=["gap"]).assign(date=panel["date"].dt.strftime("%Y-%m-%d"), ltd=panel["ltd"].dt.strftime("%Y-%m-%d"))
out.to_sql("div_error_panel", conn, if_exists="replace", index=False)
print(f"→ div_error_panel {len(panel):,} 行；样本区间", panel["date"].min().date(), "~", panel["date"].max().date())
panel.groupby(["product", "slot"]).agg(样本=("err", "size"), 起=("date", "min"), 止=("date", "max")).unstack("slot")["样本"]

## 二、误差有没有系统性偏差

逐日观测的合约窗口高度重叠（今天和明天的窗口只差一天），误差有很强的自相关。
均值的标准误用 Newey-West（HAC），滞后阶数取该位次剩余期限的中位数（折算为交易日）。

In [ ]:
def hac_mean(x, lags):
    """均值及其 Newey-West t 值"""
    x = pd.Series(x).dropna()
    if len(x) < 30:
        return np.nan, np.nan
    r = sm.OLS(x.values, np.ones(len(x))).fit(cov_type="HAC", cov_kwds={"maxlags": int(lags)})
    return float(r.params[0]), float(r.tvalues[0])


LAGS = {s: max(5, round(panel.loc[panel["slot"] == s, "dte"].median() * 250 / 365)) for s in SLOT_LABEL}
rows = []
for (p, s), g in panel.sort_values("date").groupby(["product", "slot"]):
    m_pt, t_pt = hac_mean(g["err"], LAGS[s])
    m_an, t_an = hac_mean(g["err_ann"], LAGS[s])
    rows.append({"品种": PRODUCTS[p], "位次": SLOT_LABEL[s], "样本": len(g),
                 "点数误差均值": m_pt, "t(HAC)": t_pt, "年化误差均值(百分点)": m_an, "t(HAC) ": t_an,
                 "年化|误差|中位": g["err_ann"].abs().median(), "年化|误差|p95": g["err_ann"].abs().quantile(0.95)})
bias = pd.DataFrame(rows)
bias.to_sql("research_bias", conn, if_exists="replace", index=False)
print("HAC 滞后阶数（交易日）:", LAGS)
bias.round(3)

## 三、分红季与非分红季

对每个品种 × 位次，回归 `年化误差 = a + b · 分红季`（HAC 标准误）：`a` 是非分红季的平均偏差，`b` 是分红季额外的偏差。
离散程度用年化误差绝对值的中位数比较。

In [ ]:
rows = []
for (p, s), g in panel.sort_values("date").groupby(["product", "slot"]):
    g = g.dropna(subset=["err_ann"])
    X = sm.add_constant(g["season"].astype(float).values)
    r = sm.OLS(g["err_ann"].values, X).fit(cov_type="HAC", cov_kwds={"maxlags": int(LAGS[s])})
    rows.append({"品种": PRODUCTS[p], "位次": SLOT_LABEL[s],
                 "非分红季均值 a": r.params[0], "t(a)": r.tvalues[0],
                 "分红季增量 b": r.params[1], "t(b)": r.tvalues[1],
                 "非分红季|误差|中位": g.loc[~g["season"], "err_ann"].abs().median(),
                 "分红季|误差|中位": g.loc[g["season"], "err_ann"].abs().median()})
season = pd.DataFrame(rows)
season.to_sql("research_season", conn, if_exists="replace", index=False)
season.round(3)

In [ ]:
# 按月份的平均年化误差与 |误差| 中位数（三品种合并）
prof = panel.dropna(subset=["err_ann"]).groupby(["slot", "month"])["err_ann"].agg(
    均值="mean", 绝对中位=lambda x: x.abs().median()).reset_index()
colors = {"00": "#2563eb", "01": "#f59e0b", "02": "#0891b2", "03": "#9ca3af"}
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
for s in SLOT_LABEL:
    g = prof[prof["slot"] == s]
    axs[0].plot(g["month"], g["均值"], "-o", ms=3, color=colors[s], label=SLOT_LABEL[s])
    axs[1].plot(g["month"], g["绝对中位"], "-o", ms=3, color=colors[s], label=SLOT_LABEL[s])
for ax, t in zip(axs, ["平均年化误差（百分点）", "年化误差绝对值中位数（百分点）"]):
    ax.set_title(t, fontsize=10); ax.set_xticks(range(1, 13)); ax.set_xlabel("月份")
    ax.axvspan(SEASON_MONTHS[0] - 0.5, SEASON_MONTHS[-1] + 0.5, color="#fef3c7", zorder=0)
axs[0].axhline(0, color="#6b7280", lw=0.8); axs[0].legend(frameon=False, fontsize=8.5)
fig.suptitle("分红预测误差集中在分红季，近月合约的年化影响最大", x=0.01, ha="left", fontsize=12.5, fontweight="bold")
fig.text(0.01, 0.89, "上证50/沪深300/中证500 合并；年化误差 = 预测−已实现分红 ÷ 指数 × 365 ÷ 剩余天数；黄色 = 5–8 月", fontsize=9, color="#6b7280")
fig.subplots_adjust(top=0.78, wspace=0.25)
fig.savefig(os.path.join(FIG_DIR, "fig7_err_by_month.png"), dpi=170, bbox_inches="tight"); plt.show()

## 四、升贴水方向的误判

已实现口径与预测口径的年化升贴水符号相反，即同一天一个说升水、一个说贴水。
很多误判发生在升贴水本身接近 0 的时候，影响不大，所以另外统计「实质性误判」：已实现口径的年化升贴水绝对值 ≥ 1%。

In [ ]:
v = panel.dropna(subset=["ann_real", "ann_pred"]).copy()
v["flip"] = np.sign(v["ann_real"]) != np.sign(v["ann_pred"])
v["flip_material"] = v["flip"] & (v["ann_real"].abs() >= MATERIAL)
flip = (v.groupby(["slot", "season"])[["flip", "flip_material"]].mean() * 100).unstack("season")
flip.columns = [f"{'误判' if a == 'flip' else '实质性误判'}%·{'分红季' if b else '非分红季'}" for a, b in flip.columns]
flip.index = [SLOT_LABEL[s] for s in flip.index]
flip.reset_index(names="位次").to_sql("research_flip", conn, if_exists="replace", index=False)
display(flip.round(2))

by_year = (v.groupby(["year", "slot"])["flip"].mean() * 100).unstack("slot").rename(columns=SLOT_LABEL)
display(by_year.round(1))
fig, ax = plt.subplots(figsize=(9, 3.4))
for s in SLOT_LABEL:
    ax.plot(by_year.index, by_year[SLOT_LABEL[s]], "-o", ms=3, color=colors[s], label=SLOT_LABEL[s])
ax.set_ylabel("%"); ax.legend(frameon=False, fontsize=8.5, ncol=4); ax.set_xticks(by_year.index)
fig.suptitle("2024 年起升贴水方向误判增多，各期限约 4%–7%", x=0.01, ha="left", fontsize=12.5, fontweight="bold")
fig.text(0.01, 0.89, "已实现口径与预测口径年化升贴水符号相反的观测占比；上证50/沪深300/中证500 合并", fontsize=9, color="#6b7280")
fig.subplots_adjust(top=0.78)
fig.savefig(os.path.join(FIG_DIR, "fig8_flip_by_year.png"), dpi=170, bbox_inches="tight"); plt.show()

## 五、敏感性 A：最终实施金额的前视

`forecast_exposure`（02）给出每个抽样日、每个合约窗口内，预测分红中依赖「已公告、但实施公告未发」金额的点数。
真实的预案→实施金额修订幅度没有历史数据，这里按 ±1% 和 ±5% 两种情形估算对年化基差的最大影响。

In [ ]:
ex = pd.read_sql("SELECT * FROM forecast_exposure", conn)
ex["date"] = pd.to_datetime(ex["date"])
ex = ex.merge(panel[["product", "date", "slot", "S", "dte", "season"]], on=["product", "date", "slot"], how="inner")
ex = ex[ex["dte"] >= MIN_DTE_ANN]
ex["share"] = np.where(ex["pred_pts"] > 0, ex["exposed_pts"] / ex["pred_pts"], 0.0)
for r in (0.01, 0.05):
    ex[f"ann_{int(r*100)}pct"] = ex["exposed_pts"] * r / ex["S"] * 365 / ex["dte"] * 100
expo = ex.groupby(["slot", "season"]).agg(
    样本=("exposed_pts", "size"), 有敞口天数占比=("exposed_pts", lambda x: (x > 0).mean() * 100),
    敞口点数中位=("exposed_pts", "median"), 敞口点数p95=("exposed_pts", lambda x: x.quantile(0.95)),
    占预测比例中位=("share", "median"),
    修订1pct年化影响p95=("ann_1pct", lambda x: x.quantile(0.95)), 修订5pct年化影响p95=("ann_5pct", lambda x: x.quantile(0.95)),
    修订5pct年化影响最大=("ann_5pct", "max")).reset_index()
expo["slot"] = expo["slot"].map(SLOT_LABEL); expo["season"] = expo["season"].map({True: "分红季", False: "非分红季"})
expo.to_sql("research_exposure", conn, if_exists="replace", index=False)
expo.round(4)

## 六、敏感性 B：ETF 分红公告提前天数

ETF 期权隐含基差需要加回窗口内的 ETF 自身分红，当前假设除息前 5 个交易日公告。
这里改成 0（除息前一律不加回）、3、10、20 个交易日，以及「全知」（任何时候都加回，含前视），
看 ETF 期权 − 期货（预测分红口径）的偏离如何变化。股指期权版不依赖这项假设。

In [ ]:
dist = pd.read_sql("SELECT * FROM etf_dist_detected", conn)
dist["ex_date"] = pd.to_datetime(dist["ex_date"])
NOTICE = [0, 3, 5, 10, 20, "全知"]
rows = []
for p in PRODUCTS:
    bp = b[b["product"] == p].sort_values("date").reset_index(drop=True)
    tdays = pd.DatetimeIndex(bp["date"])
    dp = dist[dist["product"] == p]
    ratio = bp["spot"] / bp["etf_close"]
    for s in SLOT_LABEL:
        base = bp[f"{s}_opt_minus_fut_pred"]
        oe = bp["date"] + pd.to_timedelta(bp[f"{s}_opt_DTE"] - 1, unit="D")
        def added(n):
            tot = np.zeros(len(bp))
            for ex_d, amt in zip(dp["ex_date"], dp["amount"]):
                known = ex_d if n == "全知" else tdays[max(tdays.searchsorted(ex_d) - n, 0)]
                inwin = (bp["date"] < ex_d) & (ex_d <= oe) & ((bp["date"] >= known) | (n == "全知"))
                tot += np.where(inwin, amt, 0.0)
            return tot
        a5 = added(5)
        for n in NOTICE:
            x = (base + (added(n) - a5) * ratio).dropna()
            rows.append({"product": p, "slot": s, "N": str(n), "n_obs": len(x), "abs_median": x.abs().median(),
                         "abs_p95": x.abs().quantile(0.95), "share_gt20": (x.abs() > 20).mean() * 100})
etfn = pd.DataFrame(rows)
etfn.to_sql("research_etf_notice", conn, if_exists="replace", index=False)
summ = etfn.groupby("N", sort=False).apply(lambda g: pd.Series({
    "|ETF期权−期货| 中位": np.average(g["abs_median"], weights=g["n_obs"]),
    "|ETF期权−期货| p95": np.average(g["abs_p95"], weights=g["n_obs"]),
    "偏离>20点的天数%": np.average(g["share_gt20"], weights=g["n_obs"])}), include_groups=False)
summ.round(2)

## 七、小结

下面打印正文引用的关键数字，结论写在研究报告里。

In [ ]:
pool = panel.dropna(subset=["err_ann"])
for s in SLOT_LABEL:
    g = pool[pool["slot"] == s]
    print(f"{SLOT_LABEL[s]}：年化误差均值 {g['err_ann'].mean():+.3f}，分红季 |误差| 中位 {g.loc[g.season,'err_ann'].abs().median():.3f}，"
          f"非分红季 {g.loc[~g.season,'err_ann'].abs().median():.3f}，误判 {((np.sign(g.ann_real)!=np.sign(g.ann_pred)).mean()*100):.1f}%")